In [7]:
import sys
!{sys.executable} -m pip install scikit-learn --break-system-packages


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [12]:
# df = pd.read_csv("Feature_Engineering_Data.csv")
df = pd.read_csv("Feature_Engineering_Data.csv")


In [13]:
# 1. בחירת הפיצ'רים שקובעים את ה"אופי" של המניה
# אלו המדדים שעל בסיסם נחליט אם מניה היא תנודתית מאוד או קצת
clustering_features = [
    'Abs_Pct_Change',         # עוצמת תנועה יומית
    'Shadow_to_Range',        # רמת ה"רעש" והזנבות (חוסר החלטיות)
    'gap',                    # קפיצות בין ימי מסחר
    'Vol_Instability',        # חוסר יציבות בווליום
    'Rolling_Volatility_10D',  # תנודתיות היסטורית
    'Dist_From_MA20'           # מרחק מהממוצע (מתיחת מחיר)
]

In [14]:
# 2. יצירת טבלת הממוצעים (זה הלב של הקובץ הזה)
# אנחנו מורידים את ה-NaN (הימים הראשונים) ומחשבים ממוצע לכל מניה
avg_for_each_stock = df.dropna().groupby('Name')[clustering_features].mean()

In [15]:
# 3. שלב הנרמול (Scaling)
# כדי שפיצ'ר אחד (כמו ווליום) לא ישתלט על המודל רק בגלל שהמספרים שלו גדולים
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(avg_for_each_stock[clustering_features])
scaled_df = pd.DataFrame(scaled_data, columns=clustering_features, index=avg_for_each_stock.index)

In [16]:
# 2. יישום משקלים ידני
scaled_df['Abs_Pct_Change'] *= 4
scaled_df['Rolling_Volatility_10D'] *= 4
scaled_df['gap'] *= 1.5
scaled_df['Dist_From_MA20'] *= 1
scaled_df['Vol_Instability'] *= 1
scaled_df['Shadow_to_Range'] *= 3

In [17]:
# 4. הרצת הסיווג (Clustering) ל-3 קבוצות
kmeans = KMeans(n_clusters=3, random_state=42)
avg_for_each_stock['Cluster_ID'] = kmeans.fit_predict(scaled_df)

In [20]:

# 5. סידור הקלסטרים לפי רמת תנודתיות (כדי ש-0 תמיד יהיה "קצת" ו-2 תמיד יהיה "מאוד")
risk_order = avg_for_each_stock.groupby('Cluster_ID')['Abs_Pct_Change'].mean().sort_values().index
mapping = {
    risk_order[0]: '0',#'תנודתי קצת'
    risk_order[1]: '1',#'תנודתי רגיל'
    risk_order[2]: '2' #'תנודתי מאוד'
}
avg_for_each_stock['Classification'] = avg_for_each_stock['Cluster_ID'].map(mapping)
avg_for_each_stock.drop(columns=['Cluster_ID'], inplace=True)
# הצגת התוצאה הסופית ב-VS Code
print("סיווג המניות הושלם:")
print(avg_for_each_stock[['Classification']].head(20))

df['Classification'] = df['Name'].map(avg_for_each_stock['Classification'])
df.to_csv("Feature_Engineering_Data_with_Classification.csv", index=False)

סיווג המניות הושלם:
     Classification
Name               
A                 1
AAL               2
AAP               1
AAPL              1
ABBV              1
ABC               0
ABT               0
ACN               0
ADBE              1
ADI               1
ADM               1
ADP               0
ADS               1
ADSK              1
AEE               0
AEP               0
AES               1
AET               1
AFL               0
AGN               1


In [21]:

# אופציונלי: שמירה לקובץ CSV כדי שיהיה לך מילון מוכן
avg_for_each_stock.to_csv('final_stock_classification.csv')

In [23]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

df=avg_for_each_stock


feature_cols = [
    "Abs_Pct_Change",
    "Shadow_to_Range",
    "gap",
    "Vol_Instability",
    "Rolling_Volatility_10D",
    "Dist_From_MA20"
]

X = df[feature_cols]

y = df["Classification"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9893617021276596
              precision    recall  f1-score   support

           0       0.98      1.00      0.99        50
           1       1.00      0.97      0.99        35
           2       1.00      1.00      1.00         9

    accuracy                           0.99        94
   macro avg       0.99      0.99      0.99        94
weighted avg       0.99      0.99      0.99        94



In [22]:
# import pandas as pd
# import matplotlib.pyplot as plt

# # 1. טעינת הנתונים
# df_raw = pd.read_csv('Cleaned_data.csv')
# df_class = pd.read_csv('final_stock_classification.csv')

# # הכנת הנתונים: המרת תאריך וחישוב תשואה יומית לכל מניה
# df_raw['date'] = pd.to_datetime(df_raw['date'])
# df_raw = df_raw.sort_values(['Name', 'date'])
# df_raw['Daily_Return'] = df_raw.groupby('Name')['close'].pct_change()

# # 2. חיבור הסיווג לנתוני המסחר
# df_combined = df_raw.merge(df_class[['Name', 'Classification']], on='Name', how='inner')

# # 3. חלוקה ל-3 קבוצות והדפסת כל שמות המניות
# print("--- רשימת מניות מלאה לפי קבוצות ---")
# for category in ['תנודתי מאוד', 'תנודתי רגיל', 'תנודתי קצת']:
#     stock_list = df_class[df_class['Classification'] == category]['Name'].unique().tolist()
#     print(f"\nקבוצה: {category} ({len(stock_list)} מניות):")
#     print(", ".join(stock_list))

# # 4. הכנת הנתונים לגרף - תשואה מצטברת לכל קבוצה
# portfolio_daily = df_combined.groupby(['date', 'Classification'])['Daily_Return'].mean().reset_index()

# # חישוב צמיחה מצטברת של דולר אחד
# portfolio_daily['Cumulative_Growth'] = portfolio_daily.groupby('Classification')['Daily_Return'].transform(lambda x: (1 + x.fillna(0)).cumprod())

# # --- תוספת: חישוב והדפסת התשואה הסופית באחוזים ---
# print("\n--- תשואה מצטברת סופית לכל קבוצה (5 שנים) ---")
# final_returns = portfolio_daily.groupby('Classification')['Cumulative_Growth'].last()
# for category, final_value in final_returns.items():
#     percentage_gain = (final_value - 1) * 100
#     print(f"{category}: {percentage_gain:.2f}%")

# # 5. יצירת הגרף
# plt.figure(figsize=(12, 7))
# pivot_df = portfolio_daily.pivot(index='date', columns='Classification', values='Cumulative_Growth')

# for column in pivot_df.columns:
#     plt.plot(pivot_df.index, pivot_df[column], label=column, linewidth=2.5)

# plt.title('השוואת ביצועים: צמיחת $1 לפי קבוצת תנודתיות (5 שנים)', fontsize=15)
# plt.ylabel('ערך התיק ($)', fontsize=12)
# plt.xlabel('תאריך', fontsize=12)
# plt.legend(title='סיווג', fontsize=10)
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()